# Clase 5 — Integración de fuentes y creación de características
**Taller de Ciencia de Datos (EIN146A25) · 2026-2 · Jueves 3 de septiembre**

Cuatro ejercicios sobre un mini-caso de **emisiones vehiculares del Gran Concepción** (dos fuentes que se parecen mucho a las reales: emisiones tipo RETC y permisos de circulación tipo INE). Al final, la plantilla del notebook `02-preprocesamiento` que deben aplicar a su propio proyecto hoy mismo (bloques 5–6).

| # | Ejercicio | Qué se practica | Min |
|---|---|---|---|
| 1 | Apilar años con esquema distinto | `concat`, alinear columnas, columna de origen, llave con tipo unificado | 15 |
| 2 | Cruzar por nombre con tabla de homologación | normalizar llaves, `merge` + `indicator`, anti-join | 20 |
| 3 | Granularidad y cardinalidad | agregar antes de unir, `validate`, fan-out, sumas de control | 20 |
| 4 | Tres características en diez minutos | razones, variaciones, indicadores | 10 |

Reglas del notebook:
- Los archivos de práctica **se generan solos** en `data_practica/` — no necesitan descargar nada.
- Las celdas marcadas **✏️ TU TURNO** son las que deben completar.
- Cada ejercicio trae su **verificación automática** (debe imprimir todos los ✓) y una **solución propuesta** plegada: intenten primero sin mirarla.
- Si un ejercicio no les sale, el siguiente puede usar la tabla de referencia `df_solN` que deja la celda del profesor.

In [ ]:
# Setup — ejecutar primero
import sys
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

CARPETA = Path("data_practica")
CARPETA.mkdir(exist_ok=True)

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("Carpeta de trabajo:", CARPETA.resolve())

In [ ]:
# Genera los archivos de práctica (no modificar esta celda)
rng = np.random.default_rng(2026)

COMUNAS = [  # código único territorial (CUT) y nombre oficial, provincia de Concepción
    ("08101", "Concepción"), ("08102", "Coronel"), ("08103", "Chiguayante"),
    ("08104", "Florida"), ("08105", "Hualqui"), ("08106", "Lota"),
    ("08107", "Penco"), ("08108", "San Pedro de la Paz"), ("08109", "Santa Juana"),
    ("08110", "Talcahuano"), ("08111", "Tomé"), ("08112", "Hualpén"),
]
CONTAMINANTES = ["CO", "NOx", "MP2.5", "COV"]
FUENTES = ["Vehículos livianos", "Vehículos pesados"]

# --- Emisiones (estilo RETC): un archivo por año, y el de 2024 cambió de esquema ---
def tabla_emisiones(anio):
    filas = []
    for cod, nom in COMUNAS:
        base = rng.uniform(40, 400)
        for cont in CONTAMINANTES:
            for fte in FUENTES:
                ton = round(base * rng.uniform(0.2, 1.5) * (1 + 0.03 * (anio - 2022)), 3)
                filas.append({
                    "anio": anio, "id_comuna": cod, "comuna": nom, "contaminante": cont,
                    "fuente": fte, "tipo_fuente": "Móvil", "toneladas": ton, "unidad": "t/año",
                    "region": "Biobío", "rubro": "Transporte en ruta",
                    "latitud": round(-36.8 + rng.uniform(-0.2, 0.2), 4),
                    "longitud": round(-73.0 + rng.uniform(-0.2, 0.2), 4),
                })
    return pd.DataFrame(filas)

em22, em23, em24 = tabla_emisiones(2022), tabla_emisiones(2023), tabla_emisiones(2024)
TOTAL_TON = round(float(em22["toneladas"].sum() + em23["toneladas"].sum() + em24["toneladas"].sum()), 3)

# 2022 y 2023: el código se publicó como número → pierde el cero (8101 en vez de 08101)
for df_, anio in ((em22, 2022), (em23, 2023)):
    df_["id_comuna"] = df_["id_comuna"].astype(int)
    df_.to_csv(CARPETA / f"emisiones_{anio}.csv", index=False)

# 2024: la fuente renombró dos columnas y eliminó las coordenadas (10 columnas en vez de 12)
em24 = em24.drop(columns=["latitud", "longitud"]).rename(columns={"id_comuna": "codigo_comuna", "toneladas": "ton"})
em24.to_csv(CARPETA / "emisiones_2024.csv", index=False)

# --- Permisos de circulación (estilo INE): nombres de comuna escritos "a mano" ---
NOMBRE_INE = {
    "Concepción": "CONCEPCION", "Coronel": "Coronel ", "Chiguayante": "chiguayante",
    "Florida": "Florida", "Hualqui": "Hualqui", "Lota": "LOTA", "Penco": "Penco",
    "San Pedro de la Paz": "San Pedro de la Paz", "Santa Juana": "Santa  Juana",
    "Talcahuano": "Talcahuano", "Tomé": "Tome", "Hualpén": "HUALPEN",
}
filas = []
for anio in (2022, 2023, 2024):
    for cod, nom in COMUNAS:
        for tipo, escala in (("Automóvil", 1.0), ("Camioneta", 0.35), ("Camión", 0.08)):
            n = int(rng.uniform(800, 9000) * escala)
            filas.append({
                "Año": anio, "Glosa Comuna": NOMBRE_INE[nom], "Tipo Vehiculo": tipo,
                "Bencina": int(n * rng.uniform(0.55, 0.8)),
                "Diesel": int(n * rng.uniform(0.15, 0.4)),
                "Electrico": int(n * rng.uniform(0.0, 0.03)),
            })
permisos = pd.DataFrame(filas)
permisos.to_csv(CARPETA / "permisos_circulacion.csv", index=False)
TOTAL_VEH = int(permisos[["Bencina", "Diesel", "Electrico"]].to_numpy().sum())

# --- Tabla de homologación: código CUT ↔ nombre oficial (con tres comunas de otras provincias) ---
cut = pd.DataFrame(COMUNAS + [("08201", "Lebu"), ("08203", "Cañete"), ("08301", "Los Ángeles")],
                   columns=["codigo_cut", "nombre_comuna"])
cut["region"] = "Biobío"
cut.to_csv(CARPETA / "comunas_cut.csv", index=False)

for f in sorted(CARPETA.glob("*.csv")):
    print(f"{f.name:28s} {f.stat().st_size:6d} bytes")
print(f"\nSuma de control — toneladas totales: {TOTAL_TON:,.3f} · vehículos totales: {TOTAL_VEH:,}")

---
## Ejercicio 1 — Apilar tres años con esquema distinto (≈15 min)

Tienen `emisiones_2022.csv`, `emisiones_2023.csv` y `emisiones_2024.csv`. Son la misma fuente, pero **el archivo 2024 cambió de esquema**: dos columnas se renombraron y las coordenadas desaparecieron. Además, en 2022 y 2023 el código de comuna se publicó como número y perdió el cero a la izquierda (`8101` en vez de `08101`).

Esto le pasa a cualquiera que apila periodos de un portal público (RETC, INE, DEIS). Su misión: **una sola tabla, con las columnas alineadas, la llave con el mismo tipo en todas las filas y una columna que diga de qué archivo salió cada fila.**

### 1.1 Primero, el error
`pd.concat` no se queja cuando los esquemas no calzan: simplemente rellena con `NaN` y sigue. Miren cuántas columnas quedan y por qué.

In [ ]:
crudos = {anio: pd.read_csv(CARPETA / f"emisiones_{anio}.csv") for anio in (2022, 2023, 2024)}
for anio, d in crudos.items():
    print(anio, d.shape, "→", list(d.columns))

apilado_ingenuo = pd.concat(crudos.values())
print("\nConcat ingenuo:", apilado_ingenuo.shape, "columnas — ¿cuántas deberían ser?")
apilado_ingenuo.isna().sum()

### 1.2 ✏️ TU TURNO
Construyan `df1` de modo que cumpla **todo** esto:

1. 288 filas (96 por año) y **13 columnas**: las 12 del esquema 2022/2023 + `origen` (el año del archivo del que salió cada fila).
2. Las columnas de 2024 **renombradas** al esquema canónico (`codigo_comuna → id_comuna`, `ton → toneladas`); `latitud` y `longitud` quedan `NaN` para 2024 — eso es correcto y se documenta, no se inventa.
3. `id_comuna` como **texto de 5 caracteres** en todas las filas (`"08101"`), sin importar cómo venía en cada archivo.
4. La suma de `toneladas` es exactamente la de los tres archivos (`TOTAL_TON`): apilar no puede crear ni perder emisiones.

Pistas: `dtype={"id_comuna": str, "codigo_comuna": str}` al leer · `.rename(columns=...)` · `.astype(str).str.zfill(5)` · `pd.concat(..., ignore_index=True)` · una columna `origen` antes de concatenar.

In [ ]:
# ✏️ TU TURNO — completen
# CANONICAS = list(crudos[2022].columns)
# partes = []
# for anio in (2022, 2023, 2024):
#     d = pd.read_csv(CARPETA / f"emisiones_{anio}.csv", dtype={...})
#     d = d.rename(columns={...})
#     d["origen"] = anio
#     partes.append(d)
# df1 = pd.concat(partes, ignore_index=True)[CANONICAS + ["origen"]]
# df1["id_comuna"] = ...
# df1.info()

### ✅ Verificación automática
Ejecuten esta celda cuando tengan su `df1`. Debe imprimir 4 ✓.

In [ ]:
CANONICAS = ["anio", "id_comuna", "comuna", "contaminante", "fuente", "tipo_fuente",
             "toneladas", "unidad", "region", "rubro", "latitud", "longitud"]

def verificar_ej1(df):
    checks = [
        ("288 filas × 13 columnas (12 canónicas + origen)",
         df.shape == (288, 13) and set(df.columns) == set(CANONICAS + ["origen"])),
        ("id_comuna es texto de 5 caracteres en todas las filas",
         pd.api.types.is_string_dtype(df["id_comuna"]) and (df["id_comuna"].str.len() == 5).all()),
        ("latitud es NaN exactamente en las 96 filas de 2024",
         int(df["latitud"].isna().sum()) == 96 and df.loc[df["latitud"].isna(), "origen"].eq(2024).all()),
        ("La suma de toneladas se conserva (TOTAL_TON)",
         pd.api.types.is_numeric_dtype(df["toneladas"]) and abs(float(df["toneladas"].sum()) - TOTAL_TON) < 1e-3),
    ]
    for nombre, ok in checks:
        print(("✓" if ok else "✗"), nombre)
    return all(ok for _, ok in checks)

# verificar_ej1(df1)

<details>
<summary><b>Solución propuesta (Ejercicio 1)</b> — clic para desplegar</summary>

```python
RENOMBRES_2024 = {"codigo_comuna": "id_comuna", "ton": "toneladas"}
partes = []
for anio in (2022, 2023, 2024):
    d = pd.read_csv(CARPETA / f"emisiones_{anio}.csv", dtype={"id_comuna": str, "codigo_comuna": str})
    d = d.rename(columns=RENOMBRES_2024)
    d["origen"] = anio
    partes.append(d)
df1 = pd.concat(partes, ignore_index=True)[CANONICAS + ["origen"]]
df1["id_comuna"] = df1["id_comuna"].astype(str).str.zfill(5)
verificar_ej1(df1)
```

La decisión que va al informe: *"2024 no trae coordenadas; se apila con `NaN` en `latitud`/`longitud` (96 filas) y no se usan esas columnas en el análisis"*. Eso es una fila del registro de decisiones.
</details>

In [ ]:
# (Celda del profesor / autocomprobación del notebook)
_partes = []
for _anio in (2022, 2023, 2024):
    _d = pd.read_csv(CARPETA / f"emisiones_{_anio}.csv", dtype={"id_comuna": str, "codigo_comuna": str})
    _d = _d.rename(columns={"codigo_comuna": "id_comuna", "ton": "toneladas"})
    _d["origen"] = _anio
    _partes.append(_d)
df_sol1 = pd.concat(_partes, ignore_index=True)[CANONICAS + ["origen"]]
df_sol1["id_comuna"] = df_sol1["id_comuna"].astype(str).str.zfill(5)
assert verificar_ej1(df_sol1), "La solución de referencia del Ej. 1 no pasa la verificación"
df_sol1.groupby("origen").size()

---
## Ejercicio 2 — Cruzar por nombre con una tabla de homologación (≈20 min)

Los permisos de circulación (`permisos_circulacion.csv`) **no traen código de comuna**: solo el nombre, escrito como lo escribió cada municipio (`CONCEPCION`, `Coronel ` con espacio al final, `chiguayante`, `Tome` sin tilde…). Las emisiones sí traen código. Para cruzarlas necesitan pasar de nombre a código con la tabla oficial `comunas_cut.csv`.

Este es **el** problema de integración en datos chilenos: la llave existe, pero no está escrita igual en las dos fuentes.

### 2.1 Primero, el error
Un `merge` por nombre tal cual. Cuenten cuántas filas encuentran su código y cuántas quedan huérfanas.

In [ ]:
permisos = pd.read_csv(CARPETA / "permisos_circulacion.csv")
cut = pd.read_csv(CARPETA / "comunas_cut.csv", dtype={"codigo_cut": str})
print("permisos:", permisos.shape, "· cut:", cut.shape)

ingenuo = permisos.merge(cut, left_on="Glosa Comuna", right_on="nombre_comuna", how="left", indicator=True)
print(ingenuo["_merge"].value_counts())
print("\nNombres que NO encontraron código:")
print(sorted(ingenuo.loc[ingenuo["_merge"] == "left_only", "Glosa Comuna"].unique()))

### 2.2 ✏️ TU TURNO
Construyan `df2` = permisos **con su `codigo_cut`**, cumpliendo:

1. Las **108 filas originales** se conservan (ni una más, ni una menos: un join no debe cambiar la granularidad).
2. `codigo_cut` sin nulos, **12 códigos distintos**, todos texto de 5 caracteres que empiezan con `08`.
3. Lo hacen **normalizando una llave auxiliar en ambas tablas** (no editando el CSV a mano, no con 12 `replace` escritos uno por uno): sin espacios sobrantes, mayúsculas uniformes, sin tildes.
4. Diagnostican con `indicator=True` y un **anti-join**: las comunas de `cut` que no aparecen en permisos (las que encuentra el anti-join) son tres, y es correcto que no aparezcan — están fuera del Gran Concepción.

Pistas: la función `normalizar` de abajo (complétenla) · `.str.strip()`, `.str.upper()`, `" ".join(s.split())` · `unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()` · `merge(..., how="left", validate="many_to_one", indicator=True)`.

In [ ]:
# ✏️ TU TURNO — completen
def normalizar(s: str) -> str:
    '''"  Santa  Juana " → "SANTA JUANA" · "Tomé" → "TOME" · "chiguayante" → "CHIGUAYANTE"'''
    ...

# permisos["llave"] = permisos["Glosa Comuna"].map(normalizar)
# cut["llave"] = cut["nombre_comuna"].map(normalizar)
# df2 = permisos.merge(cut[["llave", "codigo_cut"]], on="llave", how="left", validate="many_to_one", indicator=True)
# print(df2["_merge"].value_counts())
# huerfanas_cut = cut[~cut["llave"].isin(permisos["llave"])]   # anti-join: en cut pero no en permisos
# huerfanas_cut

### ✅ Verificación automática

In [ ]:
def verificar_ej2(df):
    codigos = df["codigo_cut"].astype(str) if "codigo_cut" in df else pd.Series([], dtype=str)
    checks = [
        ("Se conservan las 108 filas de permisos", len(df) == 108),
        ("codigo_cut sin nulos", "codigo_cut" in df and df["codigo_cut"].notna().all()),
        ("12 códigos distintos, texto de 5 caracteres que empieza con 08",
         codigos.nunique() == 12 and codigos.str.fullmatch(r"08\d{3}").all()),
        ("Todas las filas hicieron match (si usaron indicator, no queda left_only)",
         ("_merge" not in df) or (df["_merge"] == "both").all()),
    ]
    for nombre, ok in checks:
        print(("✓" if ok else "✗"), nombre)
    return all(ok for _, ok in checks)

# verificar_ej2(df2)

<details>
<summary><b>Solución propuesta (Ejercicio 2)</b> — clic para desplegar</summary>

```python
def normalizar(s: str) -> str:
    s = " ".join(str(s).split()).upper()                      # espacios sobrantes + mayúsculas
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()   # sin tildes

permisos["llave"] = permisos["Glosa Comuna"].map(normalizar)
cut["llave"] = cut["nombre_comuna"].map(normalizar)
df2 = permisos.merge(cut[["llave", "codigo_cut"]], on="llave", how="left",
                     validate="many_to_one", indicator=True)
print(df2["_merge"].value_counts())
huerfanas_cut = cut[~cut["llave"].isin(permisos["llave"])]
print(huerfanas_cut[["codigo_cut", "nombre_comuna"]])   # Lebu, Cañete, Los Ángeles: fuera del alcance
verificar_ej2(df2)
```

`validate="many_to_one"` es el seguro: si la tabla de homologación tuviera un nombre repetido, el merge fallaría en vez de duplicar filas en silencio.
</details>

In [ ]:
# (Celda del profesor / autocomprobación del notebook)
def _normalizar(s):
    s = " ".join(str(s).split()).upper()
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()

_permisos = pd.read_csv(CARPETA / "permisos_circulacion.csv")
_cut = pd.read_csv(CARPETA / "comunas_cut.csv", dtype={"codigo_cut": str})
_permisos["llave"] = _permisos["Glosa Comuna"].map(_normalizar)
_cut["llave"] = _cut["nombre_comuna"].map(_normalizar)
df_sol2 = _permisos.merge(_cut[["llave", "codigo_cut"]], on="llave", how="left",
                          validate="many_to_one", indicator=True)
assert verificar_ej2(df_sol2), "La solución de referencia del Ej. 2 no pasa la verificación"
assert len(_cut[~_cut["llave"].isin(_permisos["llave"])]) == 3
df_sol2.head()

---
## Ejercicio 3 — Granularidad y cardinalidad (≈20 min)

Ahora sí: emisiones (Ej. 1) contra permisos (Ej. 2). Pero **una fila de emisiones es comuna-año-contaminante-fuente** y **una fila de permisos es comuna-año-tipo de vehículo**. Si las cruzan tal cual, cada fila de un lado se pega con todas las del otro que compartan comuna: el resultado tiene más filas que cualquiera de las dos tablas y las toneladas se multiplican. Ese *fan-out* es el error más silencioso de toda la integración, porque **no lanza ninguna excepción**.

La regla: **agregar cada fuente a la granularidad de la pregunta antes de unir**, y unir con `validate`.

### 3.1 Primero, el error
Cruce ingenuo por comuna solamente. Cuenten las filas y comparen la suma de toneladas con `TOTAL_TON`.

In [ ]:
em = df_sol1.copy()        # o su df1
pe = df_sol2.copy()        # o su df2
pe = pe.rename(columns={"Año": "anio", "codigo_cut": "id_comuna"})

fanout = em.merge(pe, on="id_comuna", how="inner")
filas_naive = len(fanout)
print(f"emisiones {len(em)} filas × permisos {len(pe)} filas → cruce ingenuo: {filas_naive} filas")
print(f"toneladas en el cruce: {fanout['toneladas'].sum():,.0f}  vs  TOTAL_TON: {TOTAL_TON:,.0f}")

### 3.2 ✏️ TU TURNO
Construyan `df3` con **una fila por comuna-año** (36 filas) que cumpla:

1. Columnas `id_comuna`, `anio`, `toneladas` (suma de todas las emisiones de esa comuna-año), `vehiculos` (Bencina + Diesel + Electrico de todos los tipos) y `diesel` (solo Diesel).
2. El merge se hace con `validate="one_to_one"` sobre `["id_comuna", "anio"]`: si alguna agregación quedó mal, pandas debe reclamar.
3. **Sumas de control:** `toneladas` suma `TOTAL_TON` y `vehiculos` suma `TOTAL_VEH`. Agregar y unir no puede crear ni perder nada.
4. Primera característica: `ton_por_vehiculo = toneladas / vehiculos`.

Pistas: `groupby([...], as_index=False).agg(...)` en cada lado · crear `vehiculos` en permisos antes de agrupar · `merge(on=[...], how="inner", validate="one_to_one")`.

In [ ]:
# ✏️ TU TURNO — completen
# em_ca = em.groupby(["id_comuna", "anio"], as_index=False)["toneladas"].sum()
# pe["vehiculos"] = ...
# pe_ca = pe.groupby(["id_comuna", "anio"], as_index=False).agg(vehiculos=("vehiculos", "sum"), diesel=("Diesel", "sum"))
# df3 = em_ca.merge(pe_ca, on=[...], how="inner", validate="one_to_one")
# df3["ton_por_vehiculo"] = ...
# print(df3.shape); df3.head()

### ✅ Verificación automática

In [ ]:
def verificar_ej3(df):
    cols = {"id_comuna", "anio", "toneladas", "vehiculos", "diesel", "ton_por_vehiculo"}
    checks = [
        ("36 filas (12 comunas × 3 años) con las columnas pedidas",
         len(df) == 36 and cols.issubset(df.columns)),
        ("Una sola fila por comuna-año (sin fan-out)",
         {"id_comuna", "anio"}.issubset(df.columns) and not df.duplicated(subset=["id_comuna", "anio"]).any()),
        ("Sumas de control: toneladas = TOTAL_TON y vehiculos = TOTAL_VEH",
         {"toneladas", "vehiculos"}.issubset(df.columns)
         and abs(float(df["toneladas"].sum()) - TOTAL_TON) < 1e-3 and int(df["vehiculos"].sum()) == TOTAL_VEH),
        ("ton_por_vehiculo = toneladas / vehiculos, positiva en todas las filas",
         cols.issubset(df.columns) and (df["ton_por_vehiculo"] > 0).all()
         and np.allclose(df["ton_por_vehiculo"], df["toneladas"] / df["vehiculos"])),
    ]
    for nombre, ok in checks:
        print(("✓" if ok else "✗"), nombre)
    return all(ok for _, ok in checks)

# verificar_ej3(df3)

<details>
<summary><b>Solución propuesta (Ejercicio 3)</b> — clic para desplegar</summary>

```python
em_ca = em.groupby(["id_comuna", "anio"], as_index=False)["toneladas"].sum()
pe["vehiculos"] = pe["Bencina"] + pe["Diesel"] + pe["Electrico"]
pe_ca = pe.groupby(["id_comuna", "anio"], as_index=False).agg(vehiculos=("vehiculos", "sum"),
                                                              diesel=("Diesel", "sum"))
df3 = em_ca.merge(pe_ca, on=["id_comuna", "anio"], how="inner", validate="one_to_one")
df3["ton_por_vehiculo"] = df3["toneladas"] / df3["vehiculos"]
verificar_ej3(df3)
```

Lo que va al informe: *"emisiones agregadas de 288 → 36 filas (comuna-año); permisos de 108 → 36; cruce 1:1 sin pérdida (36/36), sumas de control conservadas"*. Con esa frase cualquier lector sabe que el join no inventó datos.
</details>

In [ ]:
# (Celda del profesor / autocomprobación del notebook)
_em_ca = em.groupby(["id_comuna", "anio"], as_index=False)["toneladas"].sum()
_pe = pe.copy()
_pe["vehiculos"] = _pe["Bencina"] + _pe["Diesel"] + _pe["Electrico"]
_pe_ca = _pe.groupby(["id_comuna", "anio"], as_index=False).agg(vehiculos=("vehiculos", "sum"), diesel=("Diesel", "sum"))
df_sol3 = _em_ca.merge(_pe_ca, on=["id_comuna", "anio"], how="inner", validate="one_to_one")
df_sol3["ton_por_vehiculo"] = df_sol3["toneladas"] / df_sol3["vehiculos"]
assert verificar_ej3(df_sol3), "La solución de referencia del Ej. 3 no pasa la verificación"
assert filas_naive == 288 * 9, "el cruce ingenuo debería tener 2.592 filas (fan-out 9×)"
df_sol3.sort_values("ton_por_vehiculo", ascending=False).head()

---
## Ejercicio 4 — Tres características en diez minutos (≈10 min; si no alcanza, queda para el proyecto)

Una característica nueva **se justifica con la pregunta de datos**, no con "podría servir". Sobre `df3` (o `df_sol3`) creen tres, una de cada tipo:

| Tipo | Característica | Qué responde |
|---|---|---|
| Razón / proporción | `pct_diesel = diesel / vehiculos` | ¿Qué parte del parque es diésel? (composición, no tamaño) |
| Temporal | `variacion_ton` = variación porcentual de `toneladas` respecto del **año anterior de la misma comuna** | ¿Empeora o mejora? |
| Indicador con umbral | `alta_emision = ton_por_vehiculo > mediana(ton_por_vehiculo)` | La definición operacional de "comuna crítica" |

Cumplan:
1. `pct_diesel` entre 0 y 1, sin nulos.
2. `variacion_ton` es `NaN` **solo** en 2022 (no hay año anterior): exactamente 12 nulos.
3. `alta_emision` es booleana y marca exactamente 18 de las 36 filas.

Pistas: ordenar por `["id_comuna", "anio"]` antes de calcular · `groupby("id_comuna")["toneladas"].shift(1)` · `df["x"].median()`.

In [ ]:
# ✏️ TU TURNO — completen
# df4 = df3.sort_values(["id_comuna", "anio"]).reset_index(drop=True)   # o df_sol3
# df4["pct_diesel"] = ...
# anterior = df4.groupby("id_comuna")["toneladas"].shift(1)
# df4["variacion_ton"] = ...
# df4["alta_emision"] = ...
# df4.head(6)

### ✅ Verificación automática

In [ ]:
def verificar_ej4(df):
    checks = [
        ("pct_diesel entre 0 y 1, sin nulos",
         "pct_diesel" in df and df["pct_diesel"].notna().all() and df["pct_diesel"].between(0, 1).all()),
        ("variacion_ton: exactamente 12 nulos, todos en 2022",
         "variacion_ton" in df and int(df["variacion_ton"].isna().sum()) == 12
         and df.loc[df["variacion_ton"].isna(), "anio"].eq(2022).all()),
        ("alta_emision booleana con 18 filas marcadas",
         "alta_emision" in df and pd.api.types.is_bool_dtype(df["alta_emision"]) and int(df["alta_emision"].sum()) == 18),
    ]
    for nombre, ok in checks:
        print(("✓" if ok else "✗"), nombre)
    return all(ok for _, ok in checks)

# verificar_ej4(df4)

<details>
<summary><b>Solución propuesta (Ejercicio 4)</b> — clic para desplegar</summary>

```python
df4 = df3.sort_values(["id_comuna", "anio"]).reset_index(drop=True)
df4["pct_diesel"] = df4["diesel"] / df4["vehiculos"]
anterior = df4.groupby("id_comuna")["toneladas"].shift(1)
df4["variacion_ton"] = (df4["toneladas"] - anterior) / anterior
df4["alta_emision"] = df4["ton_por_vehiculo"] > df4["ton_por_vehiculo"].median()
verificar_ej4(df4)
```

Ojo con el umbral: "mayor que la mediana" es una definición operacional **que ustedes eligieron**. En el informe se escribe el umbral, por qué ese y no otro, y cuántas filas quedan de cada lado.
</details>

In [ ]:
# (Celda del profesor / autocomprobación del notebook)
df_sol4 = df_sol3.sort_values(["id_comuna", "anio"]).reset_index(drop=True)
df_sol4["pct_diesel"] = df_sol4["diesel"] / df_sol4["vehiculos"]
_ant = df_sol4.groupby("id_comuna")["toneladas"].shift(1)
df_sol4["variacion_ton"] = (df_sol4["toneladas"] - _ant) / _ant
df_sol4["alta_emision"] = df_sol4["ton_por_vehiculo"] > df_sol4["ton_por_vehiculo"].median()
assert verificar_ej4(df_sol4), "La solución de referencia del Ej. 4 no pasa la verificación"
df_sol4.head(6)

---
## 5 · Ahora con SU proyecto — plantilla de `02-preprocesamiento.ipynb` (bloques 5–6)

Lo que sigue no es un ejercicio: es el **esqueleto** del segundo notebook de su repositorio, el que revisa el Informe 2 (10-sep). Cópienlo a `notebooks/02-preprocesamiento.ipynb` y completen hoy las tres tablas con lo que ya saben de su proyecto; el código de limpieza e integración se escribe sobre ellas.

```
proyecto/
├── README.md
├── data/
│   ├── raw/          ← intocable (Clase 2)
│   └── processed/    ← lo que produce ESTE notebook
└── notebooks/
    ├── 01-obtencion.ipynb
    └── 02-preprocesamiento.ipynb   ← crudo → limpio, reproducible, con conteos
```

Tres reglas que se corrigen en el Informe 2:
1. **Cada decisión deja rastro:** problema → celda que lo evidencia → decisión → filas afectadas → justificación. Sin conteo no hay decisión, hay opinión.
2. **Cada tabla y cada cifra del informe cita la celda que la produce** (`ver 02-preprocesamiento.ipynb, celda 14`), y los notebooks se suben **ejecutados**.
3. **Cada característica nueva se justifica desde la pregunta de datos** de la Clase 1 (unidad de análisis, variable objetivo, decisión que alimenta).

In [ ]:
# ==========================================================
# PLANTILLA 02-preprocesamiento — completar con SU proyecto
# ==========================================================

# --- 1. Mapa de integración (una fila por cruce o apilado) ---
MAPA_INTEGRACION = [
    {"fuente_a": "",              # ej: "INE permisos 2022-2024"
     "fuente_b": "",              # ej: "RETC emisiones 2022-2024"
     "operacion": "",            # "apilar" | "cruzar" | "enriquecer"
     "llave": "",                # ej: "codigo_cut + anio"
     "granularidad_final": "",   # ej: "comuna-año"
     "tipo_join": "",            # inner | left | outer
     "filas_a": None, "filas_b": None, "filas_resultado": None, "sin_match": None},
]

# --- 2. Registro de decisiones de limpieza (una fila por decisión) ---
REGISTRO_DECISIONES = [
    {"problema": "",            # ej: "10.094 id_comuna con espacio en blanco"
     "evidencia": "",           # ej: "01-obtencion.ipynb, celda 9"
     "decision": "",            # ej: "convertir a NaN e imputar desde nombre de comuna"
     "filas_afectadas": None,
     "justificacion": ""},      # por qué esa y no otra (y qué alternativa descartaron)
]

# --- 3. Características candidatas (3 a 5) ---
FEATURES = [
    {"nombre": "",              # ej: "ton_por_vehiculo"
     "formula": "",             # ej: "toneladas / vehiculos"
     "tipo": "",                # razón | temporal | agregada por grupo | indicador/umbral | derivada de fecha
     "que_responde": ""},       # qué parte de la pregunta de datos alimenta
]

for nombre, tabla in (("Mapa de integración", MAPA_INTEGRACION),
                      ("Registro de decisiones", REGISTRO_DECISIONES),
                      ("Características", FEATURES)):
    print(f"\n=== {nombre} ===")
    display(pd.DataFrame(tabla))

print("\nPlantilla lista — completar con el proyecto del equipo.")

### Checklist de salida de la Clase 5

- [ ] Ejercicios 1, 2 y 3 pasan su verificación automática (el 4 puede quedar para la casa)
- [ ] Mapa de integración del proyecto completo (fuentes, llave, granularidad, tipo de join, conteos)
- [ ] Registro de decisiones iniciado con los hallazgos del Informe 1 (cada uno con su celda)
- [ ] 3 características candidatas, cada una justificada desde la pregunta de datos
- [ ] `02-preprocesamiento.ipynb` creado en el repo y escribe a `data/processed/` (raw intacto)
- [ ] Ejecutaron el repositorio **del otro equipo** y anotaron qué falló

**Encargo para la Clase 6 (10-sep):** el notebook crudo → limpio corriendo de punta a punta. El **Informe 2 se entrega ese mismo día a las 23:59** vía AULA. Recuerden que el miércoles 9 tienen Certamen 1 de Machine Learning: el informe no se deja para el miércoles.